# Notebook17 full JEPA runner
Purpose: paired fresh models under equal active training time, using the same selected input preparation and original JEPA predictor/masks/EMA/optimizer schedule. Inputs: immutable one-candidate plan and audited Food101 TRAIN/development bundle. Outputs: full-state checkpoints, durable active-wall journals, converged initial/final probes. Default screen: four pairs,15min training eacharm, at most3h parent run including evaluation. Checkpoints: checkpoints/<run_id>/stages/<worker>/<arm>/. Resume: Run All in17_Fixed_Kernel_Feasibility_and_Learning.ipynb with unchanged run ID/profile; READY timer resumes exactly, IN_FLIGHT is UNKNOWN fairness and never resets. Long workers require detached tmux.

In [ ]:
"""Source-bound fixed-cost kernel runner; equal active-wall full JEPA and exact resume."""
from pathlib import Path
import copy,gc,json,math,os,time
import nbformat,torch
_SELF=Path(globals().get('__file__','/home/benjamin/workspace/ahr_go/fmm_jeppa/experiments/17_fixed_kernel/runner.ipynb'))
_ns={'__file__':str(_SELF.with_name('pipeline.ipynb'))}
for _cell in nbformat.read(_SELF.with_name('pipeline.ipynb'),4).cells:
    if _cell.cell_type=='code' and 'ahr-export' in _cell.metadata.get('tags',[]):exec(compile(_cell.source,str(_SELF.with_name('pipeline.ipynb')),'exec'),_ns)
C=type('Pipeline',(),_ns)
RESUME_NOTEBOOK='notebooks/17_Fixed_Kernel_Feasibility_and_Learning.ipynb'
RESUME_CELL='launch-fixed-kernel-study'
VARIANTS=('vit','optional_geometry')

def modules(root):return C.modules(root)

def build_model(root,config,kind,selected):
    runner,encoder,operator=modules(root);model=runner._default_build_model(config,kind)
    if kind=='semantic':
        options={k:v for k,v in dict(selected).items() if k in('rank','feature_seed','local_radius','spatial_mode')}
        model=operator.replace_encoder_mixers(model,options)
    return model

def builder(root,selected):return lambda config,kind:build_model(root,config,kind,selected)

def bind(root,selected):
    runner,encoder,operator=modules(root);identity=C.source_hashes(root)
    runner.configure_input_pipeline=lambda config:C.install_input_pipeline(int(config.get('input_pipeline',{}).get('workers',0)),root)
    runner.bind_model_builder(builder(root,selected),identity)
    runner.RESUME_NOTEBOOK=RESUME_NOTEBOOK;runner.RESUME_CELL=RESUME_CELL
    runner._BASE.RESUME_NOTEBOOK=RESUME_NOTEBOOK;runner._BASE.RESUME_CELL=RESUME_CELL
    return runner,identity

def data_bundle(root,artifact,config,verify_images=True):
    root,artifact=Path(root).resolve(),Path(artifact).resolve();C.bootstrap(root)
    module=C.load(root,'experiments/05_scaling/preparation.ipynb','fmm17_data05')
    def paths(source,folder,stage):
        folder=Path(folder).resolve();relative=folder.relative_to(root/'artifacts/notebook_17');run_id=relative.parts[0]
        if not run_id or any(not(c.isalnum() or c in '-_')for c in run_id):raise ValueError('Safe17 run ID required')
        return folder,root/'checkpoints'/run_id/'data',run_id
    module._stage_paths=paths
    resolution=int(config['image_size'])
    if resolution not in (128,256,512):raise ValueError('Registered17 data stages are128,256,512')
    if resolution==512:module.STAGE_DEFINITIONS['fixed512']={'id':'fixed512','image_size':512,'pretrain_pool':'selected'}
    stage={'id':('fixed512' if resolution==512 else 'resolution' if resolution==256 else 'learning'),'image_size':resolution,'pretrain_pool':'selected'}
    return module.prepare_stage_bundle(root,artifact,stage,config,verify_images=verify_images)

def _base_config(root):
    _,config,source=C.learned_source(root,'vit');return config

def make_plan(root,run_id,proposal,selected,devices=('cuda:0','cuda:1'),stage_id='screen',input_workers=0,deadline=None,gate_receipts=None):
    """Freeze one candidate and optimizer horizon; only variant differs within each fresh pair."""
    root=Path(root).resolve();base=_base_config(root);evaluation=C.load(root,'experiments/15_vit_geometry/evaluation.ipynb','fmm17_plan_evaluation')
    probe=evaluation.probe_configuration(base.get('probe_convex'));seeds=list(map(int,proposal['seeds']));updates=int(proposal.get('schedule_updates',4000));arms=[]
    if not seeds or len(seeds)!=len(set(seeds)):raise ValueError('Unique fresh paired seeds required')
    if int(input_workers)not in(0,4,8):raise ValueError('Frozen pipeline workers0,4,8 required')
    if not devices or len(devices)!=len(set(devices)) or len(devices)>len(seeds):raise ValueError('Unique explicit devices with at least one paired seed each required')
    for number,seed in enumerate(seeds):
        for variant in VARIANTS:
            cfg=copy.deepcopy(base);cfg.update(image_size=int(proposal.get('image_size',256)),device=str(devices[number%len(devices)]),dtype='float32',cpu_threads=2)
            cfg['seeds'].update(model_init=seed,data_order=seed+101,augmentation=seed+201,mask=seed+301)
            cfg['training'].update(updates=updates,microbatch=16,accumulation_steps=8,checkpoint_every=50)
            hard=float(proposal.get('hard_limit_seconds',10800));cfg['budget'].update(max_seconds=hard,target_seconds=hard,round_updates=50)
            cfg['input_pipeline']={'workers':int(input_workers),'ordered':True,'prefetch':False};cfg['execution_optimization']=False
            cfg['evaluation']['heldout_enabled']=False;cfg['evaluation']['batch_size']=8 if cfg['image_size']==512 else 32;cfg['probe_convex']=copy.deepcopy(probe)
            cfg['kernel15'].update(variant=variant,temperature_trainable=False,spatial_length_trainable=False,spatial_coefficient_trainable=False)
            cfg.pop('kernel14',None)
            arms.append({'id':'s'+str(seed)+'-'+variant,'seed':seed,'variant':variant,'condition':variant,'kind':'vit' if variant=='vit' else 'semantic','config':cfg})
    main=root/RESUME_NOTEBOOK;code=[''.join(c['source']) for c in json.loads(main.read_text())['cells'] if c['cell_type']=='code'] if main.exists() else None
    sources=C.source_hashes(root)
    return {'schema':'notebook17-equal-active-wall-plan/v1','run_id':run_id,'stage_id':stage_id,'seeds':seeds,'registered_seeds':seeds,'variants':list(VARIANTS),'arms':arms,
        'updates':updates,'milestones':[updates],'round_updates':50,'effective_batch':128,'probe_endpoints':['initial','final'] if stage_id=='screen' else ['final'],
        'probe_convex':probe,'train_count':10100,'probe_train_count':10100,'hard_limit_seconds':float(proposal.get('hard_limit_seconds',10800)),
        'active_wall_seconds_per_arm':float(proposal['active_wall_seconds_per_arm']),'wall_tolerance_fraction':.02,'wall_tolerance_seconds':2.,
        'active_wall_scope':'Each invocation including input preparation, model load, synchronization, checkpoint/metrics writes and cleanup; downtime excluded only after committed READY journal',
        'crash_policy':'IN_FLIGHT on restart fails UNKNOWN fairness; model checkpoints remain exactly resumable but unmeasured wall time is never forgiven',
        'selected':C.canonical(selected),'input_workers':int(input_workers),'source_hashes':sources,'gate_receipts':gate_receipts or [],'deadline_unix':deadline,
        'resume_notebook':RESUME_NOTEBOOK,'resume_cell':RESUME_CELL,'consumer_notebook_sha256':C.digest(code) if code is not None else None,'official_test':'CLOSED',
        'devices':list(map(str,devices)),'gpu_workers':len(devices),
        'candidate_display':'Fixed positive feature summaries with exact local correction; internal label optional_geometry is retained for15paired initialization compatibility'}

def make_screen_plan(root,run_id,selected,devices=('cuda:0','cuda:1'),deadline=None,input_workers=0,gate_receipts=None):
    resolution=int(selected.get('image_size',256))
    if resolution not in(256,512):raise ValueError('Screen target resolution must be256or512')
    proposal={'seeds':[20261901,20261902,20261903,20261904],'schedule_updates':4000,'active_wall_seconds_per_arm':900.,'hard_limit_seconds':14400 if resolution==512 else 10800,'image_size':resolution}
    return make_plan(root,run_id,proposal,selected,devices,'screen',input_workers,deadline,gate_receipts)


def worker_plan(plan,worker):
    devices=list(plan.get('devices',[]));count=int(plan.get('gpu_workers',len(devices)))
    if not devices or len(devices)!=len(set(devices)) or count!=len(devices):raise ValueError('Immutable unique device/worker membership required')
    if not isinstance(worker,str) or not worker.startswith('worker') or not worker[6:].isdigit():raise ValueError('Explicit workerN identifier required')
    index=int(worker[6:])
    if index>=count:raise ValueError('Worker index exceeds registered available devices')
    child=copy.deepcopy(plan);seeds=plan['seeds'][index::count]
    if not seeds:raise ValueError('Registered worker has no paired seeds')
    child['stage_id']=worker;child['seeds']=seeds;child['arms']=[a for a in plan['arms']if int(a['seed'])in seeds]
    if any(str(a['config']['device'])!=str(devices[index]) for a in child['arms']):raise ValueError('Worker/arm device differs from registered paired device')
    child['worker_index']=index;child['worker_device']=str(devices[index])
    return child

def checkpoint_record(runner,directory):
    manifest=runner.validate_training_manifest(directory);path=Path(directory)/manifest['latest'];record=C.file_record(path)
    return manifest,record

def timer_binding(plan,arm):
    return {'plan_sha256':C.digest(plan),'arm_sha256':C.digest(arm),'source_hashes':plan['source_hashes'],'budget_seconds':float(plan['active_wall_seconds_per_arm'])}

def validate_timer(timer,binding,manifest=None,record=None):
    if timer['binding']!=binding:raise ValueError('Registered equal-wall timer changed')
    if timer['state']=='IN_FLIGHT':raise RuntimeError('UNKNOWN active-wall fairness: invocation interrupted before atomic timer commitment; no time reset allowed')
    if timer['state']not in('READY','COMPLETE','UNKNOWN'):raise ValueError('Invalid timer state')
    if not math.isfinite(timer['active_seconds'])or timer['active_seconds']<0:raise ValueError('Invalid accumulated active time')
    if timer.get('checkpoint'):
        if record!=timer['checkpoint']or manifest['successful_updates']!=timer['successful_updates']:raise ValueError('Timer/model publication differs; UNKNOWN fairness, never restart')
    elif manifest is not None:raise ValueError('Unaccounted checkpoint exists before timed training')
    return timer

def train_arm(root,artifact,plan,bundle,arm,deadline_unix,max_invocations=None):
    root,artifact=Path(root),Path(artifact);runner,sources=bind(root,plan['selected'])
    if sources!=plan['source_hashes']:raise ValueError('Frozen17 sources changed')
    C.environment(arm['config']);folder,directory=runner._arm_paths(root,artifact,plan,arm)
    timer_path=directory/'active_wall_timer.json';binding=timer_binding(plan,arm);directory.mkdir(parents=True,exist_ok=True)
    timer=C.read(timer_path)if timer_path.exists()else {'schema':'notebook17-active-wall-journal/v1','binding':binding,'state':'READY','active_seconds':0.,'successful_updates':0,'checkpoint':None,'invocations':[],'fair_wall_budget_verified':False}
    manifest,record=checkpoint_record(runner,directory)if(directory/'manifest.json').exists()else(None,None)
    validate_timer(timer,binding,manifest,record)
    if not timer_path.exists():C.atomic_json(timer_path,timer)
    budget=float(binding['budget_seconds']);tolerance=max(float(plan['wall_tolerance_seconds']),budget*float(plan['wall_tolerance_fraction']))
    if timer['state']=='COMPLETE':return final_arm(runner,root,artifact,plan,arm,timer)
    if timer['state']=='UNKNOWN':return {'status':'UNKNOWN','arm_id':arm['id'],'reason':timer['reason'],'fair_wall_budget_verified':False,'timer':C.file_record(timer_path)}
    protocol=C.read(folder/'protocol.json')
    while timer['active_seconds']<budget:
        C.check_deadline(deadline_unix,5.)
        remaining=budget-timer['active_seconds'];before=int(timer['successful_updates'])
        if before>=int(plan['updates']):
            timer.update(state='UNKNOWN',reason='Scheduler update cap reached before registered active-wall budget; time parity unverified');C.atomic_json(timer_path,timer)
            return {'status':'UNKNOWN','arm_id':arm['id'],'reason':timer['reason'],'fair_wall_budget_verified':False}
        invocation_deadline=min(float(deadline_unix),time.time()+remaining)
        marker={'sequence':len(timer['invocations'])+1,'started_unix':time.time(),'before_step':before,'before_checkpoint':timer['checkpoint'],'remaining_seconds':remaining}
        timer.update(state='IN_FLIGHT',in_flight=marker);C.atomic_json(timer_path,timer)
        started=time.perf_counter()
        try:
            result=runner._run_stage_model(plan,arm['kind'],arm['config'],bundle,folder,directory,protocol,
                model=runner._initial_model(root,artifact,plan,arm),stop_after=min(int(plan['updates']),before+int(plan['round_updates'])),
                deadline_unix=invocation_deadline)
            manifest,record=checkpoint_record(runner,directory)
            if manifest['successful_updates']!=result['resume']['final_step']or manifest['attempted_updates']!=manifest['successful_updates']:raise ValueError('Nonfinite or failed updates invalidate timed confirmation')
            performance=copy.deepcopy(result['performance']);del result;gc.collect()
            elapsed=time.perf_counter()-started
            marker.update(ended_unix=time.time(),active_seconds=elapsed,after_step=manifest['successful_updates'],after_checkpoint=record,
                performance=performance)
            timer['active_seconds']+=elapsed;timer['successful_updates']=manifest['successful_updates'];timer['checkpoint']=record;timer['invocations'].append(marker)
            timer.pop('in_flight',None);timer['state']='READY';C.atomic_json(timer_path,timer)
            
        except BaseException:
            # Keep IN_FLIGHT explicitly. A full model checkpoint may be valid, but active wall between publications cannot be recovered honestly.
            raise
        if timer['active_seconds']>=budget:break
        if max_invocations is not None and len(timer['invocations'])>=int(max_invocations):
            return {'status':'READY_FOR_EXACT_RESUME','arm_id':arm['id'],'successful_updates':timer['successful_updates'],'timer':C.file_record(timer_path),'fair_wall_budget_verified':False}
    fair=budget<=timer['active_seconds']<=budget+tolerance and timer['successful_updates']<int(plan['updates'])
    timer.update(state='COMPLETE'if fair else'UNKNOWN',fair_wall_budget_verified=fair,overshoot_seconds=max(0.,timer['active_seconds']-budget),
        reason='Registered equal active-wall budget reached'if fair else'Active-wall overshoot/update cap exceeded registered tolerance')
    C.atomic_json(timer_path,timer);return final_arm(runner,root,artifact,plan,arm,timer)

def final_arm(runner,root,artifact,plan,arm,timer):
    folder,directory=runner._arm_paths(root,artifact,plan,arm);manifest,record=checkpoint_record(runner,directory)
    validate_timer(timer,timer_binding(plan,arm),manifest,record)
    return {'status':'COMPLETE'if timer['fair_wall_budget_verified']else'UNKNOWN','arm_id':arm['id'],'seed':arm['seed'],'variant':arm['variant'],'model_id':arm['kind'],
        'actual_updates':manifest['successful_updates'],'attempted_updates':manifest['attempted_updates'],'image_exposures':manifest['image_exposures'],
        'latest_checkpoint':record,'source_encoder':record,'initial_teacher':C.read(artifact/'stage_registration.json')['arms'][arm['id']]['initial_teacher'],
        'fair_wall_budget_verified':timer['fair_wall_budget_verified'],'active_wall_seconds':timer['active_seconds'],'timer':C.file_record(directory/'active_wall_timer.json'),
        'schedule_updates':plan['updates'],'selection':'Last registered wall-budget checkpoint; never best-dev/epoch selected'}

def train_worker(root,artifact,plan,bundle,deadline_unix):
    C.bootstrap(root)
    from fmm_jepa.food101_training import run_writer
    root,artifact=Path(root),Path(artifact);runner,sources=bind(root,plan['selected']);C.environment(plan['arms'][0]['config'])
    if len(plan['arms'])*float(plan['active_wall_seconds_per_arm'])>900 and not os.environ.get('TMUX'):raise RuntimeError('Long17 worker must run in detached tmux through the frontdoor')
    if sources!=plan['source_hashes']:raise ValueError('17 source binding changed')
    runner.prepare_stage(root,artifact,plan,bundle);preflight=runner.verify_stage_resume(root,artifact,plan,bundle,deadline_unix)
    if preflight['status']!='PASS':raise ValueError('Actual17 full-state exact-resume preflight failed')
    path=artifact/'training_result.json';saved=C.read(path)if path.exists()else {'status':'RUNNING','plan_hash':C.digest(plan),'arms':{},'fair_wall_budget_verified':False,'gate_receipts':plan['gate_receipts']}
    if saved['plan_hash']!=C.digest(plan):raise ValueError('Timed worker plan changed')
    with run_writer(artifact/'timed_training_writer'):
        for arm in plan['arms']:
            saved['arms'][arm['id']]=train_arm(root,artifact,plan,bundle,arm,deadline_unix);C.atomic_json(path,saved)
            if saved['arms'][arm['id']]['status']!='COMPLETE':saved['status']='UNKNOWN';C.atomic_json(path,saved);return saved
    values=list(saved['arms'].values());tolerance=max(plan['wall_tolerance_seconds'],plan['active_wall_seconds_per_arm']*plan['wall_tolerance_fraction'])
    fair=all(v['fair_wall_budget_verified']for v in values)and max(v['active_wall_seconds']for v in values)-min(v['active_wall_seconds']for v in values)<=tolerance
    saved.update(status='COMPLETE'if fair else'UNKNOWN',fair_wall_budget_verified=fair,preflight=C.file_record(artifact/'stage_preflight.json'))
    C.atomic_json(path,saved);return saved


def evaluate_worker(root,artifact,plan,bundle,training,deadline_unix):
    """Converged initial+final readouts for actual wall-budget update counts; official data closed."""
    root,artifact=Path(root),Path(artifact);C.environment(plan['arms'][0]['config']);C.install_input_pipeline(int(plan['input_workers']),root)
    module=C.load(root,'experiments/15_vit_geometry/evaluation.ipynb','fmm17_timed_evaluation')
    module.bind_model_builder(builder(root,plan['selected']),plan['source_hashes']);rows=[];initial_rows=[]
    for arm in plan['arms']:
        saved=training['arms'][arm['id']]
        if saved['status']!='COMPLETE':return {'status':'UNKNOWN','rows':rows,'reason':'Incomplete timed training; test remains closed'}
        endpoints=[]
        if 'initial' in plan.get('probe_endpoints',[]):endpoints.append(('initial',saved['initial_teacher'],0,None))
        endpoints.append(('final',saved['source_encoder'],saved['actual_updates'],{'active_wall_seconds':saved['active_wall_seconds']}))
        for endpoint,record,count,cost in endpoints:
            C.check_deadline(deadline_unix)
            row=module.evaluate_endpoint(root,artifact/'evaluation',arm,bundle,record,count,deadline_unix,probe_config=plan['probe_convex'],endpoint=endpoint,
                training_cost=cost,run_id=plan['run_id'],stage_id=plan['stage_id'])
            (initial_rows if endpoint=='initial' else rows).append(row);C.atomic_json(artifact/'evaluation_result.json',{'status':'RUNNING','rows':rows,'initial_rows':initial_rows,'official_test':'CLOSED'})
            if row.get('status')!='COMPLETE' or not row.get('converged') or not row.get('numerical_valid',False):
                result={'status':'UNKNOWN','rows':rows,'initial_rows':initial_rows,'reason':'Readout convergence/numerical validity not certified','official_test':'CLOSED'};C.atomic_json(artifact/'evaluation_result.json',result);return result
    result={'status':'COMPLETE','rows':rows,'initial_rows':initial_rows,'official_test':'CLOSED','source_hashes':plan['source_hashes'],'plan_hash':C.digest(plan)};C.atomic_json(artifact/'evaluation_result.json',result);return result


def preflight(root,artifact,plan,bundle,deadline):
    backend,sources=bind(root,plan['selected'])
    if sources!=plan['source_hashes']:raise ValueError('Actual17 sources differ before preflight')
    backend.prepare_stage(root,artifact,plan,bundle)
    return backend.verify_stage_resume(root,artifact,plan,bundle,deadline)


def actual_preflight(root,artifact,selected,image_size=256,workers=0,deadline=None,device='cuda:0'):
    """Discarded actual GPU/config4-vs2+2 smoke, bounded600s; no learning/accuracy claim.

    Call after all source exports are frozen. Each resolution receives a separate
    unique artifact/run ID and discarded seed. Image preparation reads only TRAIN
    and development verification bytes; no readout or official test is opened.
    Reinvocation honors the original deadline and validates saved full state.
    """
    root,artifact=Path(root).resolve(),Path(artifact).resolve();artifact.mkdir(parents=True,exist_ok=True)
    if not artifact.is_relative_to(root/'artifacts/notebook_17'):raise ValueError('Actual preflight artifact belongs underartifacts/notebook_17/<unique_run_id>')
    relative=artifact.relative_to(root/'artifacts/notebook_17');run_id=relative.parts[0];resolution=int(image_size)
    if resolution not in(256,512):raise ValueError('Actual targetresolution256or512 required')
    request_path=artifact/'actual_preflight_request.json';selected={**C.canonical(selected),'image_size':resolution}
    original=C.read(request_path) if request_path.exists() else None
    if deadline is None:deadline=float(original['deadline_unix']) if original else time.time()+600.
    request={'schema':'notebook17-actual-gpu-preflight/v1','run_id':run_id,'resolution':resolution,'input_workers':int(workers),'selected':selected,
        'device':str(device),'deadline_unix':float(deadline),'source_hashes':C.source_hashes(root),'discarded_seed':20261998 if resolution==256 else 20261999,
        'quality_selection':'FORBIDDEN; discarded exact-resume proof only','official_test':'CLOSED'}
    C.immutable(request_path,request);started=time.perf_counter()
    try:
        C.check_deadline(deadline,5.)
        proposal={'seeds':[request['discarded_seed']],'schedule_updates':4000,'active_wall_seconds_per_arm':900.,'hard_limit_seconds':600.,'image_size':resolution}
        plan=make_plan(root,run_id,proposal,selected,(str(device),),'actual-preflight'+str(resolution),int(workers),deadline,[])
        plan['probe_endpoints']=['final'];plan['evidence_role']='ACTUAL_CONFIG_DISCARDED_GPU_EXACT_RESUME_ONLY'
        C.immutable(artifact/'actual_preflight_plan.json',plan)
        bundle=data_bundle(root,artifact/('data_'+str(resolution)),plan['arms'][0]['config'],verify_images=True)
        certificate=preflight(root,artifact/('stage_'+str(resolution)),plan,bundle,deadline)
        result={'status':certificate['status'],'passed':certificate['status']=='PASS' and all(v.get('bitwise_equal',False) for v in certificate['arms'].values()),
            'certificate':certificate,'arms':certificate['arms'],'source_hashes':request['source_hashes'],'plan':C.file_record(artifact/'actual_preflight_plan.json'),
            'elapsed_seconds':time.perf_counter()-started,'image_size':resolution,'input_workers':int(workers),'scientific_training_launched':False,
            'actual_config':'FP32 dim192/depth4/head3, physical16 effective128, accumulation8,4000schedule; stop4 only',
            'quality_selection':'FORBIDDEN','official_test_pixels_read':0}
    except (RuntimeError,ValueError,TimeoutError,FloatingPointError) as error:
        result={'status':'UNKNOWN','passed':False,'error':repr(error),'source_hashes':request['source_hashes'],'elapsed_seconds':time.perf_counter()-started,
            'image_size':resolution,'input_workers':int(workers),'scientific_training_launched':False,'quality_selection':'FORBIDDEN','official_test_pixels_read':0}
    finally:
        gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    C.atomic_json(artifact/'actual_preflight_result.json',result);return result
